# 🚀 FEDHEP POST-MIDTERM EXPERIMENT SUITE ON CIFAR-100 (SUPERVISOR FEEDBACK)

This notebook implements the complete **Post-Midterm Experimental Suite** for FedHEP on **CIFAR-100** using **ResNet-9**:

1. **STUDY C (Idea C)**: **New Attacks Suite Targeting Temporal Trust Tracking (TTT)**
   - Evaluates *Inner Product Manipulation* (IPM), *Adaptive Cosine Attack* (ACA), *Delayed Onset Attack* (DOA), *ALIE*, and *Sign-Flip* at $q \in \{0.2, 0.3\}$ across 3 random seeds $\{42, 123, 7\}$.
   - Tests whether stealthy attacks that maintain positive cosine alignment bypass TTT without triggering isolation ($\tau_{\text{iso}} = 0.25$).

2. **STUDY B (Idea B)**: **Majority Attacker Stress-Test ($q \ge 50\%$)**
   - Evaluates FedHEP vs **FedAvg** vs **Ditto** across attacker fractions $q \in \{0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7\}$.
   - Identifies the empirical breakdown threshold $q^*$ where the Subspace-Constrained Cosine Filtering (SCCF) consensus centroid inverts.

3. **STUDY A (Idea A)**: **Multi-Seed Peer Cluster Count ($K$) Sensitivity Sweep**
   - Evaluates $K \in \{1, 2, 3, 5, 8, 10, 15\}$ across 3 Dirichlet heterogeneity levels ($\alpha \in \{0.05, 0.1, 0.5\}$) with 3 seeds $\{42, 123, 7\}$ (mean $\pm$ std).

---
### 📁 Dedicated Output Folder & Real-Time Auto-Save:
* **All results are written to a dedicated directory** `/kaggle/working/oreh_results/` (completely separate from the git repository code).
* **Real-time appending**: Every single completed run is flushed immediately to CSV (`mode='a'`). If Kaggle times out, re-running automatically skips finished runs and resumes instantly!
* **1-Click Download**: The final cell packages everything into `/kaggle/working/oreh_results_summary.zip` for instant download directly from Kaggle Output.


In [ ]:
import os
import sys

# 1. Repository Setup
ROOT = '/kaggle/working/Topology-aware-FDL'
if not os.path.exists(ROOT):
    print('🚀 Cloning repository from GitHub...')
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
else:
    print('🔄 Repository exists, pulling latest updates from main...')
    !cd {ROOT} && git pull origin main

%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 2. DEDICATED OUTPUT DIRECTORY (Outside of repo, directly visible in Kaggle Output)
if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working/oreh_results'
else:
    OUT_DIR = os.path.abspath('./oreh_results')

SANITY_DIR = os.path.join(OUT_DIR, 'sanity')
EVAL_DIR = os.path.join(OUT_DIR, 'eval')
FIG_DIR = os.path.join(OUT_DIR, 'figures')

os.makedirs(SANITY_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('✅ Working Directory:', os.getcwd())
print('📁 Dedicated Results Directory:', OUT_DIR)
print('📊 CSV Tables Directory:', EVAL_DIR)
print('📈 Figures Directory:', FIG_DIR)


In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy
import torch

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM Capacity: {vram_gb:.2f} GB')
    torch.backends.cudnn.benchmark = True
else:
    print('⚠️ No GPU detected! Please navigate to Notebook Settings -> Accelerator -> GPU T4 x1.')


In [ ]:
from src.data.dataset import _auto_link_dataset

data_dir = './data'
found = _auto_link_dataset(data_dir, 'cifar100')
if found:
    print('⚡ [Kaggle Input] CIFAR-100 dataset successfully verified and linked! (0.0s latency, 0 downloads)')
else:
    print('ℹ️ Note: If CIFAR-100 is attached under /kaggle/input, the experiment runner will automatically link it.')


In [ ]:
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running 1-round pre-flight sanity check on CIFAR-100 using {device}...')

sanity_runner = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=1,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=1000,
    test_subset=500,
    data_dir=data_dir,
    output_dir=SANITY_DIR,
)

for test_atk in ['ipm', 'adaptive_cosine', 'delayed_onset']:
    test_res = sanity_runner.run_single_experiment(
        defense_scope='both',
        attack_target_tier='all',
        attack_type=test_atk,
        byzantine_rate=0.2,
        seed=42,
    )
    print(f'✅ Sanity Check {test_atk.upper()} PASSED! Acc: {test_res["final_test_acc"]}% | Byz Trust: {test_res["byzantine_trust_score"]} | Latency: {test_res["total_defense_latency_ms"]} ms')

print('\n🚀 All Pre-Flight Sanity Checks Successfully Completed!')


## 🧪 STUDY C (Idea C): New Attacks Suite Targeting Temporal Trust Tracking
Evaluates whether attacks designed to maintain small positive cosine similarities (*IPM*), oscillate trust (*Adaptive Cosine*), or exploit TTT's EMA memory (*Delayed Onset*) can degrade FedHEP without being isolated.

* **Attacks**: `sign_flip`, `alie`, `ipm`, `adaptive_cosine`, `delayed_onset`
* **Byzantine Rates**: $q = 0.20$ (3 attackers), $q = 0.30$ (4–5 attackers)
* **Seeds**: $\{42, 123, 7\}$
* **Output File**: `/kaggle/working/oreh_results/eval/cifar100_postmid_attacks.csv` (incrementally saved after each run)


In [ ]:
# ==============================================================================
# Cell 6: [STUDY C] New Attacks Suite Targeting TTT
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_c = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
)

df_attacks = runner_c.run_postmid_attacks(
    attacks=['sign_flip', 'alie', 'ipm', 'adaptive_cosine', 'delayed_onset'],
    byzantine_rates=[0.2, 0.3],
    seeds=[42, 123, 7],
    defense_scope='both',
    attack_target_tier='all',
    ipm_epsilon=0.1,
    attack_onset_round=15,
)

print('\n--- STUDY C (NEW ATTACKS SUITE) SUMMARY TABLE ---')
cols_c = ['method', 'attack_type', 'byzantine_rate', 'seed', 'final_test_acc', 'bottom_10_fairness', 'byzantine_trust_score', 'honest_trust_score']
if not df_attacks.empty:
    print(df_attacks[[c for c in cols_c if c in df_attacks.columns]].to_string(index=False))


## 🛡️ STUDY B (Idea B): Majority Attacker Stress-Test ($q \ge 50\%$)
Tests what happens when malicious clients form the majority ($q \in \{0.5, 0.6, 0.7\}$). Does the Subspace-Constrained Cosine Filtering centroid invert, and does FedHEP perform worse than undefended FedAvg or Ditto?

* **Methods**: **Defended FedHEP**, **FedAvg**, **Ditto**
* **Attacker Fractions**: $q \in \{0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7\}$
* **Attacks**: `sign_flip`, `alie`
* **Seeds**: $\{42, 123, 7\}$
* **Output File**: `/kaggle/working/oreh_results/eval/cifar100_postmid_majority_q.csv` (incrementally saved after each run)


In [ ]:
# ==============================================================================
# Cell 7: [STUDY B] Majority Attacker Stress-Test (q >= 50%)
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_b = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
)

df_majority = runner_b.run_postmid_majority_q(
    byzantine_rates=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    attacks=['sign_flip', 'alie'],
    methods=['fedhep', 'fedavg', 'ditto'],
    seeds=[42, 123, 7],
    defense_scope='both',
)

print('\n--- STUDY B (MAJORITY q >= 50%) SUMMARY TABLE ---')
cols_b = ['method', 'attack_type', 'byzantine_rate', 'seed', 'final_test_acc', 'bottom_10_fairness', 'byzantine_trust_score']
if not df_majority.empty:
    print(df_majority[[c for c in cols_b if c in df_majority.columns]].to_string(index=False))


## 🌐 STUDY A (Idea A): Multi-Seed Peer Clusters ($K$) Sensitivity Sweep
Performs a rigorous statistical sweep over number of peer clusters $K \in \{1, 2, 3, 5, 8, 10, 15\}$ across Extreme ($\alpha=0.05$), Severe ($\alpha=0.1$), and Moderate ($\alpha=0.5$) skews with 3 seeds.

* **Cluster Counts**: $K \in \{1, 2, 3, 5, 8, 10, 15\}$
* **Dirichlet Alphas**: $\alpha \in \{0.05, 0.1, 0.5\}$
* **Seeds**: $\{42, 123, 7\}$
* **Output File**: `/kaggle/working/oreh_results/eval/cifar100_postmid_k_clusters.csv` (incrementally saved after each run)


In [ ]:
# ==============================================================================
# Cell 8: [STUDY A] Multi-Seed Peer Clusters (K) Sensitivity Sweep
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_a = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
)

df_k = runner_a.run_postmid_k_clusters(
    k_values=[1, 2, 3, 5, 8, 10, 15],
    alphas=[0.05, 0.1, 0.5],
    seeds=[42, 123, 7],
    byzantine_rate=0.0,
)

print('\n--- STUDY A (MULTI-SEED K SWEEP) SUMMARY TABLE ---')
cols_k = ['num_clusters', 'alpha', 'seed', 'final_test_acc', 'bottom_10_fairness', 'total_runtime_s']
if not df_k.empty:
    print(df_k[[c for c in cols_k if c in df_k.columns]].to_string(index=False))


## 📊 Publication-Quality Visualizations & Statistical Analysis
Generates the publication figures addressing the supervisor feedback:
1. `postmid_c_attacks_comparison.png`: Accuracy & tail fairness under TTT-targeting attacks.
2. `postmid_b_majority_q_curve.png`: Breakdown threshold $q^*$ curve showing centroid inversion.
3. `postmid_a_k_cluster_multiseed_curve.png`: Accuracy vs $K$ with mean $\pm$ std error bands.

* Output Directory: `/kaggle/working/oreh_results/figures/`


In [ ]:
# ==============================================================================
# Cell 9: Post-Midterm Visualizations and Statistical Analysis
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Plot Study C: Attacks Comparison
attacks_file = os.path.join(EVAL_DIR, 'cifar100_postmid_attacks.csv')
if os.path.exists(attacks_file):
    df_c = pd.read_csv(attacks_file)
    if not df_c.empty:
        plt.figure(figsize=(10, 5))
        summary_c = df_c.groupby(['attack_type', 'byzantine_rate'])['final_test_acc'].agg(['mean', 'std']).reset_index()
        atks = summary_c['attack_type'].unique()
        q_vals = summary_c['byzantine_rate'].unique()
        x = np.arange(len(atks))
        width = 0.35
        for i, q in enumerate(sorted(q_vals)):
            sub = summary_c[summary_c['byzantine_rate'] == q]
            acc_map = dict(zip(sub['attack_type'], sub['mean']))
            std_map = dict(zip(sub['attack_type'], sub['std'].fillna(0)))
            means = [acc_map.get(a, 0.0) for a in atks]
            stds = [std_map.get(a, 0.0) for a in atks]
            plt.bar(x + i * width, means, width, yerr=stds, capsize=4, label=f'q = {q}')
        plt.title('CIFAR-100: Robustness under Advanced Byzantine Attacks Targeting TTT')
        plt.xlabel('Attack Type')
        plt.ylabel('Personalized Test Accuracy (%)')
        plt.xticks(x + width / 2, [a.replace('_', ' ').title() for a in atks])
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, 'postmid_c_attacks_comparison.png'), dpi=200)
        plt.show()

# 2. Plot Study B: Majority Attacker Inversion Curve
majority_file = os.path.join(EVAL_DIR, 'cifar100_postmid_majority_q.csv')
if os.path.exists(majority_file):
    df_b = pd.read_csv(majority_file)
    if not df_b.empty:
        plt.figure(figsize=(9, 5))
        df_b_sf = df_b[df_b['attack_type'].isin(['sign_flip', 'none'])]
        summary_b = df_b_sf.groupby(['method', 'byzantine_rate'])['final_test_acc'].mean().reset_index()
        for m in summary_b['method'].unique():
            sub = summary_b[summary_b['method'] == m].sort_values('byzantine_rate')
            plt.plot(sub['byzantine_rate'], sub['final_test_acc'], marker='o', linewidth=2, label=m)
        plt.axvline(x=0.5, color='red', linestyle=':', label='q = 50% Majority Boundary')
        plt.title('CIFAR-100: Byzantine Resilience Curve up to q = 70% (Sign-Flip)')
        plt.xlabel('Attacker Fraction (q)')
        plt.ylabel('Test Accuracy (%)')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, 'postmid_b_majority_q_curve.png'), dpi=200)
        plt.show()

# 3. Plot Study A: Multi-seed K Curve with Error Bands
k_file = os.path.join(EVAL_DIR, 'cifar100_postmid_k_clusters.csv')
if os.path.exists(k_file):
    df_k = pd.read_csv(k_file)
    if not df_k.empty:
        plt.figure(figsize=(8, 5))
        summary_k = df_k.groupby(['alpha', 'num_clusters'])['final_test_acc'].agg(['mean', 'std']).reset_index()
        for a in sorted(summary_k['alpha'].unique()):
            sub = summary_k[summary_k['alpha'] == a].sort_values('num_clusters')
            k_vals = sub['num_clusters'].values
            means = sub['mean'].values
            stds = sub['std'].fillna(0).values
            label = f'alpha = {a}'
            plt.plot(k_vals, means, marker='s', linewidth=2, label=label)
            plt.fill_between(k_vals, means - stds, means + stds, alpha=0.2)
        plt.title('CIFAR-100: K-Cluster Sensitivity Sweep with 3-Seed Error Bands')
        plt.xlabel('Number of Peer Clusters (K)')
        plt.ylabel('Personalized Test Accuracy (%)')
        plt.xticks([1, 2, 3, 5, 8, 10, 15])
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, 'postmid_a_k_cluster_multiseed_curve.png'), dpi=200)
        plt.show()

print('✅ All charts successfully generated and saved to:', FIG_DIR)


## 📦 1-Click ZIP Packaging for Direct Download
Packages the dedicated `/kaggle/working/oreh_results/` directory into a single zip archive for 1-click download from the Kaggle Output sidebar!


In [ ]:
# ==============================================================================
# Cell 10: 1-Click ZIP Packaging for Direct Download
# ==============================================================================
import shutil

zip_base = '/kaggle/working/oreh_results_summary' if os.path.exists('/kaggle/working') else './oreh_results_summary'
shutil.make_archive(zip_base, 'zip', OUT_DIR)

print('🎉 PACKAGING COMPLETED!')
print(f'📁 Dedicated Folder: {OUT_DIR}')
print(f'📦 Direct Download File: {zip_base}.zip')
print('You can download oreh_results_summary.zip directly from the Kaggle Output sidebar without opening the code repo!')
